# ANALYTE — klasifikuesi XLM-R (Faza 7, E11)

1. **Runtime → Change runtime type → T4 GPU.**
2. Lokalisht: `python -m ml.colab.make_bundle` → `ml/artifacts/classifier_bundle.zip`.
3. Ekzekutoni qelizat me radhë. Ngarkoni zip-in kur kërkohet.
4. Shkarkoni `results.zip` në fund dhe ndiqni hapat lokalë në qelizën e fundit.

Trajnohen dy modele: `sentence` (vetëm fjalia, si në ADR 0009) dhe `context`
(fjalia + përmbledhja e kontekstit). Të dyja maten mbi të njëjtat 192 tekste
si E10.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
uploaded = files.upload()  # zgjidhni classifier_bundle.zip

In [ ]:
!unzip -oq classifier_bundle.zip
!cat classifier_data/meta.json

In [ ]:
# Colab-u ka torch dhe scikit-learn; transformers përditësohet për `eval_strategy`
# dhe `processing_class`. Versionet e sakta ruhen te run.json i çdo ekzekutimi.
!pip -q install -U "transformers>=4.46" accelerate

## Hyrja `sentence` — rreth 5-10 minuta në T4

In [ ]:
!python train_classifier.py --data classifier_data --input sentence --out runs/sentence

## Hyrja `context` — rreth 20-30 minuta në T4

In [ ]:
!python train_classifier.py --data classifier_data --input context --out runs/context

## Opsionale: ruajtja e modeleve në Drive

Peshat janë rreth 1 GB për model dhe nuk hyjnë në `results.zip`. Nëse do t'i
përdorni më vonë në verifikim (E9), ruajini këtu; përndryshe Colab-u i fshin
kur mbyllet sesioni.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/analyte_runs
!cp -r runs/sentence runs/context /content/drive/MyDrive/analyte_runs/

## Rezultatet — vetëm parashikimet dhe run.json, pa pesha

In [ ]:
!cd runs && zip -qr ../results.zip */run.json */predictions_*.jsonl
files.download('results.zip')

## Hapat lokalë

```bash
# hapeni results.zip te ml/artifacts/runs/
python -m ml.evaluate_classifier --run ml/artifacts/runs/sentence
python -m ml.evaluate_classifier --run ml/artifacts/runs/context
```

Rezultatet shkruhen te `evaluation/results/E11/sentence/` dhe
`evaluation/results/E11/context/`. Raporti i rrjedhjes është te
`evaluation/results/E11/leakage.json`.